# 🔍 CSCI435 — YOLOv8 Fine-Tuning Notebook
## Smart Surveillance & Safety Monitor
**University of Wollongong in Dubai**

---

### What this notebook does
1. Installs YOLOv8 (Ultralytics) and Roboflow
2. Downloads the **Hard Hat Workers** dataset (~3,000 annotated images)
3. Fine-tunes the pre-trained `yolov8n.pt` model on this dataset
4. Evaluates the trained model and shows metrics
5. Lets you download the trained weights (`best.pt`)

### Why this dataset?
The Hard Hat Workers dataset detects safety equipment on construction workers:
- `hardhat` — safety helmets
- `vest` — high-visibility vests
- `person` — workers

This fits our **Smart Surveillance & Safety Monitor** theme perfectly.
After training, rename `best.pt` → `custom_model.pt` and put it in `models/`.

---
⚠️ **Important:** Make sure you are using a **GPU runtime**.
Go to **Runtime → Change runtime type → T4 GPU** before running.

## Step 1 — Check GPU and Install Libraries

In [ ]:
# Verify GPU is available
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# Install required libraries
!pip install ultralytics roboflow --quiet

# Verify installation
import ultralytics
ultralytics.checks()
print('\n✅ Installation complete')

## Step 2 — Download the Dataset from Roboflow

You need a **free Roboflow account** to get an API key:
1. Go to https://roboflow.com and sign up (free)
2. Go to your account settings → Copy your API key
3. Paste it below where it says `YOUR_ROBOFLOW_API_KEY`

The **Hard Hat Workers** dataset has:
- ~3,000 images with YOLOv8-format annotations
- 3 classes: `hardhat`, `vest`, `person`
- Train / Validation / Test split already prepared

In [ ]:
from roboflow import Roboflow

# ─────────────────────────────────────────────────────────────────────────────
# PASTE YOUR ROBOFLOW API KEY HERE
ROBOFLOW_API_KEY = "YOUR_ROBOFLOW_API_KEY"
# ─────────────────────────────────────────────────────────────────────────────

rf      = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("roboflow-universe-projects").project("hard-hat-workers")
dataset = project.version(2).download("yolov8")

print('\n✅ Dataset downloaded to:', dataset.location)
print('Dataset name:', dataset.name)

In [ ]:
import os
import yaml

# Find the data.yaml file (YOLOv8 uses this to locate images and labels)
data_yaml = os.path.join(dataset.location, 'data.yaml')

with open(data_yaml, 'r') as f:
    config = yaml.safe_load(f)

print('Dataset configuration:')
print(f"  Classes ({config['nc']}): {config['names']}")
print(f"  Train path: {config.get('train', 'N/A')}")
print(f"  Val path:   {config.get('val',   'N/A')}")

# Count images in each split
for split in ['train', 'valid', 'test']:
    img_dir = os.path.join(dataset.location, split, 'images')
    if os.path.exists(img_dir):
        n = len(os.listdir(img_dir))
        print(f"  {split}: {n} images")

## Step 3 — Fine-Tune YOLOv8

### What is fine-tuning?
The pre-trained `yolov8n.pt` already knows 80 COCO classes (cars, people, etc.).
Fine-tuning **reuses these learned features** but adapts the final layers to our
specific 3-class task (hardhat, vest, person).

This is much faster than training from scratch (~20 mins vs. days).

### Key parameters:
- `epochs=50` — number of full passes through the training data
- `imgsz=640` — image size (YOLOv8 default)
- `batch=16` — images processed per gradient update
- `device=0` — use GPU 0

In [ ]:
from ultralytics import YOLO

# Load the base pre-trained model
# 'n' = nano (smallest, fastest — ideal for a student project demo)
model = YOLO('yolov8n.pt')

print('Starting fine-tuning on Hard Hat Workers dataset...')
print('This will take approximately 20-30 minutes on a T4 GPU.\n')

results = model.train(
    data=data_yaml,        # path to dataset config
    epochs=50,             # training epochs (more = better accuracy, longer time)
    imgsz=640,             # input image size
    batch=16,              # batch size (reduce to 8 if you get CUDA out-of-memory)
    device=0,              # GPU device index
    project='runs/detect', # output directory
    name='hardhat_train',  # run name
    patience=10,           # early stopping: stop if no improvement for 10 epochs
    save=True,             # save best and last checkpoints
    plots=True,            # generate training plots
    verbose=True,
)

print('\n✅ Training complete!')
print('Best weights saved to:', results.save_dir)

## Step 4 — Evaluate the Trained Model

YOLOv8 evaluation metrics:
- **mAP50** — mean Average Precision at IoU threshold 0.50 (standard)
- **mAP50-95** — averaged over IoU thresholds 0.50 to 0.95 (stricter)
- **Precision** — of all predicted boxes, what fraction were correct?
- **Recall** — of all actual objects, what fraction did we find?

In [ ]:
import os

# Load the best checkpoint from training
best_model_path = os.path.join(str(results.save_dir), 'weights', 'best.pt')
best_model      = YOLO(best_model_path)

# Run validation on the validation set
metrics = best_model.val(data=data_yaml, device=0)

print('\n📊 Evaluation Results:')
print(f"  mAP50:    {metrics.box.map50:.4f}")
print(f"  mAP50-95: {metrics.box.map:.4f}")
print(f"  Precision:{metrics.box.p.mean():.4f}")
print(f"  Recall:   {metrics.box.r.mean():.4f}")
print()
print('Per-class mAP50:')
class_names = ['hardhat', 'vest', 'person']
for name, ap in zip(class_names, metrics.box.ap50):
    print(f"  {name}: {ap:.4f}")

In [ ]:
# Display training curves (loss, mAP over epochs)
from IPython.display import Image, display
import os

plots = [
    'results.png',         # loss and mAP curves
    'confusion_matrix.png',# confusion matrix
    'val_batch0_pred.png', # sample predictions on validation images
]

save_dir = str(results.save_dir)

for plot in plots:
    path = os.path.join(save_dir, plot)
    if os.path.exists(path):
        print(f'\n--- {plot} ---')
        display(Image(filename=path, width=800))

## Step 5 — Run Inference on a Test Image

Test the model on a sample image to visually verify it works.

In [ ]:
import os
import glob
from IPython.display import Image, display

# Get a sample test image
test_images = glob.glob(os.path.join(dataset.location, 'test', 'images', '*.jpg'))

if not test_images:
    test_images = glob.glob(os.path.join(dataset.location, 'valid', 'images', '*.jpg'))

if test_images:
    sample = test_images[0]
    print(f'Running inference on: {sample}')

    pred_results = best_model.predict(
        source=sample,
        conf=0.5,
        save=True,
        project='/tmp',
        name='test_pred'
    )

    # Find and show the result image
    result_imgs = glob.glob('/tmp/test_pred/*.jpg')
    if result_imgs:
        display(Image(filename=result_imgs[0], width=640))
    
    print('\nDetections:')
    for box in pred_results[0].boxes:
        cls  = int(box.cls[0])
        conf = float(box.conf[0])
        print(f'  {class_names[cls]}: {conf:.3f}')
else:
    print('No test images found — check dataset path.')

## Step 6 — Download the Trained Weights

Download `best.pt`, rename it to `custom_model.pt`, and place it in the
`models/` folder of your project.

The app will automatically detect and use it next time you run `streamlit run app.py`.

In [ ]:
from google.colab import files
import shutil

# Copy best weights to a convenient location
best_weights_src = os.path.join(str(results.save_dir), 'weights', 'best.pt')
best_weights_dst = '/content/custom_model.pt'

shutil.copy(best_weights_src, best_weights_dst)
print(f'Copied weights to: {best_weights_dst}')
print('Starting download...')

# This triggers a browser download
files.download(best_weights_dst)

print('\n✅ Done!')
print('Rename the file to custom_model.pt and place it in your project models/ folder.')

## Summary — What to record for your report

Copy these values into your report's **Experiments and Results** section:

| Metric | Value |
|---|---|
| Base model | YOLOv8n (pre-trained on COCO, 80 classes) |
| Dataset | Hard Hat Workers (Roboflow Universe) |
| Training images | ~2,400 |
| Validation images | ~300 |
| Classes | hardhat, vest, person |
| Epochs trained | 50 |
| mAP50 | _(fill in from your results)_ |
| mAP50-95 | _(fill in from your results)_ |
| Training time | _(fill in — ~20 min on T4)_ |

These numbers demonstrate **quantitative evaluation** which is required by the marking criteria.

### Key talking points for your defence:
- **Why YOLOv8n?** Fastest variant, achieves real-time (>10 FPS) on CPU as required.
- **Why fine-tune?** Pre-trained COCO model doesn't recognise hardhats/vests specifically.
  Fine-tuning adapts the model to our safety domain with only ~20 min of training.
- **Why this dataset?** Matches our surveillance use-case (workplace safety monitoring).
  Publicly available, well-annotated, right size for a course project.
